# NB01: Baseline Teachers

Train Teacher-A (MIT-BIH) and Teacher-B (PTB-XL rhythm). Never touch DS2/fold 10.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab - skip Drive mount.')


## RUN MODE — edit the cell below (only switch in this notebook)

| `XFAD_DRY_RUN` | Meaning |
|---|---|
| `'1'` | **Smoke test** — tiny samples, 1 epoch, fast sanity check (~minutes) |
| `'0'` | **Full paper run** — all data, full training, both student arms (~hours) |

On a **full run**, NB02–NB05 automatically process **baseline** and **saliency** students (everything NB06 needs).

Then: **Runtime → Run all** (restart session after changing the switch).


In [ ]:
import os

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  ONLY SWITCH IN THIS NOTEBOOK                                        ║
# ║  '0' = FULL PAPER RUN (all data, both student arms)                  ║
# ║  '1' = SMOKE TEST (tiny data, baseline student only)                 ║
# ╚══════════════════════════════════════════════════════════════════════╝
os.environ['XFAD_DRY_RUN'] = '0'


In [ ]:
import os
import sys
from pathlib import Path


def bootstrap_xfad_root() -> Path:
    """Locate repo root and put it on sys.path (works in Colab + local)."""
    candidates = []
    env = os.environ.get('XFAD_ROOT')
    if env:
        candidates.append(Path(env))
    candidates.extend([
        Path('/content/drive/MyDrive/XFAD'),
        Path.cwd(),
        Path.cwd().parent,
    ])
    candidates.extend(Path.cwd().resolve().parents)
    seen = set()
    for p in candidates:
        try:
            p = p.resolve()
        except OSError:
            continue
        if p in seen:
            continue
        seen.add(p)
        if (p / 'src' / 'config.py').exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            os.environ.setdefault('XFAD_ROOT', str(p))
            return p
    raise FileNotFoundError(
        'XFAD repo root not found (expected src/config.py). '
        'In Colab: mount Drive first, then either keep the project at '
        '/content/drive/MyDrive/XFAD or set '
        "os.environ['XFAD_ROOT'] = '/content/drive/MyDrive/<your>/XFAD'."
    )

REPO_ROOT = bootstrap_xfad_root()
print('REPO_ROOT:', REPO_ROOT)

from src.utils.io import set_seed
set_seed(42)

from src.utils.io import resolve_dry_run, print_run_banner
DRY_RUN = resolve_dry_run()
print_run_banner(notebook='NB01_Teachers', dry_run=DRY_RUN)
import json
import torch
import torch.nn as nn
from src.data.loader_utils import make_loader
from src.utils.io import ensure_dir
from src.config import MODEL_DIR, RESULTS_DIR, COMMON_FS, MITBIH_DS1_TRAIN, MITBIH_DS1_VAL, PTBXL_TRAIN_FOLDS, PTBXL_VAL_FOLD, TEACHER_EPOCHS, EARLY_STOPPING_PATIENCE, USE_CLASS_WEIGHTS
from src.data.mitbih import MITBIHDataset
from src.data.ptbxl import PTBXLDataset
from src.models.teacher_a import TeacherA
from src.models.teacher_b import TeacherB

device = torch.device('cuda' if torch.cuda.is_available() and not DRY_RUN else 'cpu')
ensure_dir(MODEL_DIR)


In [ ]:
ds1_train = MITBIHDataset(MITBIH_DS1_TRAIN, dry_run=DRY_RUN, use_cache=True)
ds1_val = MITBIHDataset(MITBIH_DS1_VAL, dry_run=DRY_RUN, use_cache=True)
ptbxl_train = PTBXLDataset(PTBXL_TRAIN_FOLDS, dry_run=DRY_RUN, use_cache=True)
ptbxl_val = PTBXLDataset([PTBXL_VAL_FOLD], dry_run=DRY_RUN, use_cache=True)

batch_size = 2 if DRY_RUN else 64
mit_train_loader = make_loader(ds1_train, batch_size=batch_size, shuffle=True, dry_run=DRY_RUN)
mit_val_loader = make_loader(ds1_val, batch_size=batch_size, dry_run=DRY_RUN)
ptb_train_loader = make_loader(ptbxl_train, batch_size=batch_size, shuffle=True, dry_run=DRY_RUN)
ptb_val_loader = make_loader(ptbxl_val, batch_size=batch_size, dry_run=DRY_RUN)

x_mit, y_mit = next(iter(mit_train_loader))
x_ptb, y_ptb = next(iter(ptb_train_loader))
assert x_mit.shape[1:] == (2, COMMON_FS)
assert x_ptb.shape[1:] == (12, 10 * COMMON_FS)
assert y_ptb.ndim == 1  # single-label rhythm task
print('Data loaders ready')


In [ ]:
from src.utils.class_weights import class_weights_from_labels

model_a = TeacherA().to(device)
model_b = TeacherB().to(device)

def train_epoch(model, loader, criterion, optimizer, multilabel=False):
    model.train()
    running, n = 0.0, 0
    for i, (x, y) in enumerate(loader):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        running += float(loss.detach()); n += 1
        if DRY_RUN:
            break
    return running / max(n, 1)

def eval_loss(model, loader, criterion):
    model.eval()
    total = 0.0
    n = 0
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            x, y = x.to(device), y.to(device)
            total += criterion(model(x), y).item()
            n += 1
            if DRY_RUN:
                break
    return total / max(n, 1)

epochs = 1 if DRY_RUN else TEACHER_EPOCHS
opt_a = torch.optim.Adam(model_a.parameters(), lr=1e-3)
opt_b = torch.optim.Adam(model_b.parameters(), lr=1e-3)

if USE_CLASS_WEIGHTS and not DRY_RUN:
    w_a = class_weights_from_labels([y for _, y in ds1_train.samples], num_classes=5).to(device)
    w_b = class_weights_from_labels([y for _, y in ptbxl_train.samples], num_classes=12).to(device)
    print('Using inverse-frequency class weights for both teachers')
else:
    w_a = w_b = None

crit_a = nn.CrossEntropyLoss(weight=w_a)
crit_b = nn.CrossEntropyLoss(weight=w_b)

from src.utils.early_stopping import EarlyStopping
stop_a = EarlyStopping(patience=EARLY_STOPPING_PATIENCE)
stop_b = EarlyStopping(patience=EARLY_STOPPING_PATIENCE)
history = []

for ep in range(epochs):
    train_a = train_epoch(model_a, mit_train_loader, crit_a, opt_a)
    train_b = train_epoch(model_b, ptb_train_loader, crit_b, opt_b)
    val_a = eval_loss(model_a, mit_val_loader, crit_a)
    val_b = eval_loss(model_b, ptb_val_loader, crit_b)
    history.append({'epoch': ep + 1, 'train_a': train_a, 'train_b': train_b, 'val_a': val_a, 'val_b': val_b})
    print(f'epoch {ep+1}: train_a={train_a:.4f} train_b={train_b:.4f} | val_a={val_a:.4f} val_b={val_b:.4f}')
    done_a = stop_a.step(val_a, model_a)
    done_b = stop_b.step(val_b, model_b)
    if done_a and done_b:
        print(f'Early stopping at epoch {ep+1} (patience={EARLY_STOPPING_PATIENCE})')
        break

stop_a.restore(model_a)
stop_b.restore(model_b)
torch.save(model_a.state_dict(), os.path.join(MODEL_DIR, 'teacher_a.pt'))
torch.save(model_b.state_dict(), os.path.join(MODEL_DIR, 'teacher_b.pt'))
best_a = min(history, key=lambda r: r['val_a'])
best_b = min(history, key=lambda r: r['val_b'])
ensure_dir(RESULTS_DIR)
history_path = os.path.join(RESULTS_DIR, 'teacher_training_history.json')
with open(history_path, 'w', encoding='utf-8') as f:
    json.dump({
        'dry_run': DRY_RUN,
        'max_epochs': epochs,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'best_val_a': stop_a.best,
        'best_val_b': stop_b.best,
        'best_epoch_a': best_a['epoch'],
        'best_epoch_b': best_b['epoch'],
        'stopped_epoch': history[-1]['epoch'] if history else None,
        'epochs': history,
    }, f, indent=2)
print('Teachers saved to', MODEL_DIR, f'(best val_a={stop_a.best:.4f} epoch {best_a["epoch"]}, best val_b={stop_b.best:.4f} epoch {best_b["epoch"]})')
print('Training history saved to', history_path)


In [ ]:
from pathlib import Path
from src.config import MODEL_DIR
for name in ('teacher_a.pt', 'teacher_b.pt'):
    p = Path(MODEL_DIR) / name
    print(name, 'OK' if p.exists() else 'MISS', f'({p.stat().st_size/1e6:.1f} MB)' if p.exists() else '')